# Retrieval Testing, Search Evaluation, and Chunking Experiment Analysis

This notebook evaluates semantic retrieval on the vectors ingested into Qdrant from the structured chunk pipeline.

The notebook is responsible for:
1. verifying AWS Bedrock and Qdrant connectivity
2. embedding evaluation queries
3. running vector search against the Qdrant collection
4. applying metadata-aware filters using the updated chunk payload schema
5. inspecting retrieved results with taxonomy, section, chunk type, and context provenance
6. testing retrieval across multiple query scenarios
7. supporting chunking-strategy comparison for experiment analysis
8. saving retrieval evaluation outputs for later reporting and README documentation

This notebook is designed for:
- full-document XBRL-derived chunk collections
- multiple chunk types such as filing metadata, raw fact groups, unified metric groups, and narrative disclosures
- commercial / NBFC / banking / insurance filings
- metadata-aware retrieval using fields like taxonomy, chunk type, report type, section, source file, and context type
- offline retrieval evaluation and chunking experiment comparison

This notebook does **not** perform chunk generation, vector ingestion, or final answer generation.
It is focused only on retrieval testing, search analysis, and experiment evaluation.

## 1. Import Required Libraries

In [6]:
import json
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional
from collections import Counter

import boto3
from botocore.exceptions import ClientError

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

## 1. Define Retrieval Configuration

In [7]:
CONFIG = {
    "aws_region": "us-east-1",
    "embed_model_id": "amazon.titan-embed-text-v2:0",
    "embed_dimension": 1024,
    "embed_normalize": True,
    "embed_max_retries": 5,
    "query_text_max_chars": 4000,
    "qdrant_host": "localhost",
    "qdrant_port": 6333,
    "collection_name": "financial_chunks_xbrl_context_aware_v1",
    "default_top_k": 5,
    "result_preview_chars": 600,
    "low_score_warning_threshold": 0.45,
    "manual_search_output_path": "../data/processed/audits/retrieval_manual_search_v1.json",
    "scenario_results_output_path": "../data/processed/audits/retrieval_scenario_results_v1.json",
    "scenario_summary_output_path": "../data/processed/audits/retrieval_scenario_summary_v1.json",
    "demo_company_name_filter": None,
    "demo_taxonomy_filter": None,
}

print("Retrieval configuration loaded successfully.\n")
for key, value in CONFIG.items():
    print(f"{key}: {value}")

Retrieval configuration loaded successfully.

aws_region: us-east-1
embed_model_id: amazon.titan-embed-text-v2:0
embed_dimension: 1024
embed_normalize: True
embed_max_retries: 5
query_text_max_chars: 4000
qdrant_host: localhost
qdrant_port: 6333
collection_name: financial_chunks_xbrl_context_aware_v1
default_top_k: 5
result_preview_chars: 600
low_score_warning_threshold: 0.45
manual_search_output_path: ../data/processed/audits/retrieval_manual_search_v1.json
scenario_results_output_path: ../data/processed/audits/retrieval_scenario_results_v1.json
scenario_summary_output_path: ../data/processed/audits/retrieval_scenario_summary_v1.json
demo_company_name_filter: None
demo_taxonomy_filter: None


## 2. Define File and Utility Helpers

In [8]:
def save_json(data: Any, output_path: Path) -> Path:
    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
    return path


def count_words(text: Any) -> int:
    return len(re.findall(r"\b\w+\b", str(text)))


def normalize_whitespace(text: str) -> str:
    normalized = str(text).replace("\x00", " ").replace("\ufeff", " ")
    normalized = re.sub(r"\r\n?", "\n", normalized)
    normalized = re.sub(r"[ \t]+", " ", normalized)
    normalized = re.sub(r"\n{3,}", "\n\n", normalized)
    return normalized.strip()


def prepare_query_text(query_text: str, max_chars: int) -> Dict[str, Any]:
    original_text = str(query_text)
    normalized_text = normalize_whitespace(original_text)

    prepared_text = normalized_text
    was_truncated = False

    if len(prepared_text) > max_chars:
        truncated = prepared_text[:max_chars]
        if " " in truncated:
            truncated = truncated.rsplit(" ", 1)[0]
        prepared_text = truncated.strip()
        was_truncated = True

    return {
        "original_text": original_text,
        "prepared_text": prepared_text,
        "original_char_count": len(original_text),
        "prepared_char_count": len(prepared_text),
        "original_word_count": count_words(original_text),
        "prepared_word_count": count_words(prepared_text),
        "was_truncated": was_truncated,
    }


def safe_round(value: Optional[float], digits: int = 4) -> Optional[float]:
    if value is None:
        return None
    return round(float(value), digits)


print("Utility helpers ready.")

Utility helpers ready.


## 3. Verify AWS Credentials and Initialize Bedrock Client

In [9]:
print("Verifying AWS access...\n")

try:
    sts_client = boto3.client("sts", region_name=CONFIG["aws_region"])
    identity = sts_client.get_caller_identity()

    bedrock_client = boto3.client(
        "bedrock-runtime",
        region_name=CONFIG["aws_region"]
    )

    print("AWS verification successful.")
    print("Bedrock client initialized successfully.")

except Exception as e:
    raise RuntimeError(
        "AWS verification failed. "
        "Please check your AWS credentials, region, and Bedrock access.\n"
        f"Details: {e}"
    )

Verifying AWS access...

AWS verification successful.
Bedrock client initialized successfully.


## 4. Verify Qdrant Connectivity and Target Collection Availability

In [10]:
print("Verifying Qdrant connectivity...\n")

try:
    qdrant_client = QdrantClient(
        host=CONFIG["qdrant_host"],
        port=CONFIG["qdrant_port"]
    )

    collections_response = qdrant_client.get_collections()
    existing_collections = [collection.name for collection in collections_response.collections]

    print("Qdrant connection successful.")
    print(f"Qdrant host: {CONFIG['qdrant_host']}:{CONFIG['qdrant_port']}")
    print(f"Existing collections found: {len(existing_collections)}")

    if CONFIG["collection_name"] not in existing_collections:
        raise ValueError(
            f"Target collection '{CONFIG['collection_name']}' was not found. "
            "Please run the ingestion notebook first."
        )

    print(f"Target collection '{CONFIG['collection_name']}' is available.")

except Exception as e:
    raise ConnectionError(
        "Qdrant verification failed. "
        "Please ensure the Qdrant server is running and the target collection exists.\n"
        f"Details: {e}"
    )

Verifying Qdrant connectivity...

Qdrant connection successful.
Qdrant host: localhost:6333
Existing collections found: 2
Target collection 'financial_chunks_xbrl_context_aware_v1' is available.


## 5. Inspect Target Collection Summary

In [11]:
collection_info = qdrant_client.get_collection(CONFIG["collection_name"])

collection_summary = {
    "collection_name": CONFIG["collection_name"],
    "status": str(collection_info.status),
    "points_count": collection_info.points_count,
    "indexed_vectors_count": collection_info.indexed_vectors_count,
}

print("Collection summary")
print("-" * 80)
print(json.dumps(collection_summary, indent=2, ensure_ascii=False))

Collection summary
--------------------------------------------------------------------------------
{
  "collection_name": "financial_chunks_xbrl_context_aware_v1",
  "status": "green",
  "points_count": 1383,
  "indexed_vectors_count": 0
}


## 6. Define the Query Embedding Function with Retry Logic

In [12]:
def get_query_embedding(query_text: str, max_retries: int = 5) -> Dict[str, Any]:
    payload = {
        "inputText": query_text,
        "dimensions": CONFIG["embed_dimension"],
        "normalize": CONFIG["embed_normalize"],
    }

    body = json.dumps(payload)
    request_start_time = time.time()
    throttled_retries = 0

    for attempt in range(max_retries):
        try:
            response = bedrock_client.invoke_model(
                body=body,
                modelId=CONFIG["embed_model_id"],
                accept="application/json",
                contentType="application/json"
            )

            response_body = json.loads(response["body"].read())
            embedding = response_body.get("embedding")

            if not embedding:
                raise ValueError("Embedding response is missing the 'embedding' field.")

            if len(embedding) != CONFIG["embed_dimension"]:
                raise ValueError(
                    f"Embedding dimension mismatch. "
                    f"Expected {CONFIG['embed_dimension']}, got {len(embedding)}."
                )

            return {
                "embedding": embedding,
                "attempts_used": attempt + 1,
                "throttled_retries": throttled_retries,
                "latency_seconds": round(time.time() - request_start_time, 4),
            }

        except ClientError as e:
            error_code = e.response.get("Error", {}).get("Code", "")

            if error_code in {"ThrottlingException", "TooManyRequestsException"}:
                throttled_retries += 1
                sleep_time = 2 ** attempt
                print(
                    f"Throttling detected. Retrying in {sleep_time} seconds "
                    f"(attempt {attempt + 1}/{max_retries})..."
                )
                time.sleep(sleep_time)
            else:
                raise RuntimeError(f"Bedrock query embedding failed: {e}")

        except Exception as e:
            raise RuntimeError(f"Unexpected query embedding error: {e}")

    raise TimeoutError(f"Failed to get query embedding after {max_retries} retries.")

## 7. Run a Query Embedding Smoke Test

In [13]:
TEST_QUERY = "How much revenue did the company earn and what were the major expenses?"

prepared_test_query = prepare_query_text(
    TEST_QUERY,
    max_chars=CONFIG["query_text_max_chars"],
)

print("Running query embedding smoke test...\n")
test_embedding_result = get_query_embedding(
    prepared_test_query["prepared_text"],
    max_retries=CONFIG["embed_max_retries"],
)

print("Query embedding successful.")
print("Query text:", TEST_QUERY)
print("Prepared char count:", prepared_test_query["prepared_char_count"])
print("Prepared word count:", prepared_test_query["prepared_word_count"])
print("Was truncated:", prepared_test_query["was_truncated"])
print("Vector length:", len(test_embedding_result["embedding"]))
print("Expected length:", CONFIG["embed_dimension"])
print(f"Response time: {test_embedding_result['latency_seconds']:.4f} seconds")
print("Attempts used:", test_embedding_result["attempts_used"])
print("Sample vector values:", test_embedding_result["embedding"][:5])

Running query embedding smoke test...

Query embedding successful.
Query text: How much revenue did the company earn and what were the major expenses?
Prepared char count: 71
Prepared word count: 13
Was truncated: False
Vector length: 1024
Expected length: 1024
Response time: 6.1924 seconds
Attempts used: 1
Sample vector values: [-0.01209072582423687, 0.02426915429532528, 0.017320195212960243, 0.003370405174791813, -0.055488962680101395]


## 8. Define Metadata Filter Builder for the Updated Payload Schema

In [14]:
def build_search_filter(filters: Optional[Dict[str, Any]] = None) -> Optional[Filter]:
    if not filters:
        return None

    conditions = []

    for key, value in filters.items():
        if value is None:
            continue

        conditions.append(
            FieldCondition(
                key=key,
                match=MatchValue(value=value)
            )
        )

    if not conditions:
        return None

    return Filter(must=conditions)


print("Metadata filter builder ready.")

Metadata filter builder ready.


## 9. Define Search Execution and Result Normalization Helpers

In [15]:
def normalize_result_record(result: Any, rank: int, query_text: str) -> Dict[str, Any]:
    payload = result.payload or {}
    text_preview = payload.get("text_preview") or (
        (payload.get("text", "")[:CONFIG["result_preview_chars"]] + "...")
        if payload.get("text")
        else ""
    )

    return {
        "rank": rank,
        "point_id": str(result.id),
        "score": safe_round(result.score, 6),
        "query_text": query_text,
        "chunk_id": payload.get("chunk_id"),
        "company_name": payload.get("company_name"),
        "taxonomy": payload.get("taxonomy"),
        "report_type": payload.get("report_type"),
        "source_file": payload.get("source_file"),
        "source_file_stem": payload.get("source_file_stem"),
        "section_key": payload.get("section_key"),
        "section_name": payload.get("section_name"),
        "chunk_type": payload.get("chunk_type"),
        "reporting_period": payload.get("reporting_period"),
        "context_type": payload.get("context_type"),
        "context_ids": payload.get("context_ids", []),
        "start_date": payload.get("start_date"),
        "end_date": payload.get("end_date"),
        "instant_date": payload.get("instant_date"),
        "dimension_members": payload.get("dimension_members", []),
        "source_tags": payload.get("source_tags", []),
        "unified_fields": payload.get("unified_fields", []),
        "context_signature": payload.get("context_signature"),
        "word_count": payload.get("word_count"),
        "is_narrative_chunk": payload.get("is_narrative_chunk"),
        "is_unified_chunk": payload.get("is_unified_chunk"),
        "is_metadata_chunk": payload.get("is_metadata_chunk"),
        "has_dimensions": payload.get("has_dimensions"),
        "text_preview": text_preview,
        "full_text": payload.get("text"),
    }


def evaluate_filter_alignment(result_record: Dict[str, Any], filters: Dict[str, Any]) -> Dict[str, Any]:
    checks = {}

    for key, expected_value in (filters or {}).items():
        if expected_value is None:
            continue

        actual_value = result_record.get(key)
        checks[key] = {
            "expected": expected_value,
            "actual": actual_value,
            "match": actual_value == expected_value,
        }

    return {
        "checks": checks,
        "all_filters_matched": all(item["match"] for item in checks.values()) if checks else True,
    }


def summarize_search_results(
    normalized_results: List[Dict[str, Any]],
    filters: Dict[str, Any],
) -> Dict[str, Any]:
    scores = [item["score"] for item in normalized_results if item.get("score") is not None]

    full_filter_match_count = 0
    for item in normalized_results:
        if item.get("filter_alignment", {}).get("all_filters_matched") is True:
            full_filter_match_count += 1

    top_result = normalized_results[0] if normalized_results else {}

    return {
        "result_count": len(normalized_results),
        "top_score": max(scores) if scores else None,
        "lowest_score": min(scores) if scores else None,
        "average_score": safe_round(sum(scores) / len(scores), 6) if scores else None,
        "full_filter_match_count": full_filter_match_count,
        "filters_applied": filters or {},
        "top_result_company_name": top_result.get("company_name"),
        "top_result_section_name": top_result.get("section_name"),
        "top_result_chunk_type": top_result.get("chunk_type"),
        "top_result_taxonomy": top_result.get("taxonomy"),
    }


def execute_vector_search(
    query_text: str,
    top_k: int,
    filters: Optional[Dict[str, Any]] = None,
    scenario_name: str = "manual_search",
    scenario_description: Optional[str] = None,
) -> Dict[str, Any]:
    prepared_query = prepare_query_text(
        query_text,
        max_chars=CONFIG["query_text_max_chars"],
    )

    embedding_result = get_query_embedding(
        prepared_query["prepared_text"],
        max_retries=CONFIG["embed_max_retries"],
    )

    search_filter = build_search_filter(filters)
    search_start_time = time.time()

    raw_search_result = qdrant_client.query_points(
        collection_name=CONFIG["collection_name"],
        query=embedding_result["embedding"],
        query_filter=search_filter,
        limit=top_k,
        with_payload=True,
    )

    search_latency = round(time.time() - search_start_time, 4)

    normalized_results = []
    for rank, result in enumerate(raw_search_result.points, start=1):
        record = normalize_result_record(result, rank, prepared_query["prepared_text"])
        record["filter_alignment"] = evaluate_filter_alignment(record, filters or {})
        normalized_results.append(record)

    result_summary = summarize_search_results(normalized_results, filters or {})

    return {
        "scenario_name": scenario_name,
        "scenario_description": scenario_description,
        "query_text": query_text,
        "prepared_query_text": prepared_query["prepared_text"],
        "top_k": top_k,
        "filters": filters or {},
        "query_prep": prepared_query,
        "embedding_meta": {
            "attempts_used": embedding_result["attempts_used"],
            "throttled_retries": embedding_result["throttled_retries"],
            "latency_seconds": embedding_result["latency_seconds"],
        },
        "search_meta": {
            "search_latency_seconds": search_latency,
            "collection_name": CONFIG["collection_name"],
        },
        "summary": result_summary,
        "results": normalized_results,
    }


print("Search execution helpers ready.")

Search execution helpers ready.


## 10. Define Result Display Helper

In [16]:
def print_search_results(search_run: Dict[str, Any], max_results: int = 5) -> None:
    print(f"Scenario: {search_run['scenario_name']}")
    print(f"Description: {search_run.get('scenario_description')}")
    print(f"Query: {search_run['query_text']}")
    print(f"Top K: {search_run['top_k']}")
    print(f"Filters: {json.dumps(search_run['filters'], ensure_ascii=False)}")
    print(f"Embedding latency (s): {search_run['embedding_meta']['latency_seconds']}")
    print(f"Search latency (s): {search_run['search_meta']['search_latency_seconds']}")
    print(f"Result count: {search_run['summary']['result_count']}")
    print(f"Top score: {search_run['summary']['top_score']}")
    print("-" * 100)

    results = search_run.get("results", [])
    if not results:
        print("No results found.")
        return

    for item in results[:max_results]:
        print(f"Rank: {item['rank']} | Score: {item['score']}")
        print(f"Company Name: {item.get('company_name')}")
        print(f"Taxonomy: {item.get('taxonomy')}")
        print(f"Section: {item.get('section_name')}")
        print(f"Chunk Type: {item.get('chunk_type')}")
        print(f"Reporting Period: {item.get('reporting_period')}")
        print(f"Context Type: {item.get('context_type')}")
        print(f"Chunk ID: {item.get('chunk_id')}")
        print(f"Source File: {item.get('source_file')}")
        print(f"Unified Fields: {item.get('unified_fields')}")
        print(f"Source Tags: {item.get('source_tags')}")
        print("Text Preview:")
        print((item.get("text_preview", "")[:CONFIG["result_preview_chars"]] + "...") if item.get("text_preview") else "N/A")
        print(f"All filters matched: {item.get('filter_alignment', {}).get('all_filters_matched')}")
        print("-" * 100)


print("Result display helper ready.")

Result display helper ready.


## 11. Configure a Manual Search Example

In [17]:
MANUAL_SEARCH_QUERY = "How much revenue did the company earn and what were the major expenses?"
MANUAL_TOP_K = CONFIG["default_top_k"]

MANUAL_FILTERS = {
    "chunk_type": None,
    "taxonomy": None,
    "section_name": None,
    "report_type": None,
    "reporting_period": None,
    "context_type": None,
    "company_name": CONFIG["demo_company_name_filter"],
    "source_file": None,
    "source_file_stem": None,
    "is_unified_chunk": None,
    "is_narrative_chunk": None,
    "is_metadata_chunk": None,
    "has_dimensions": None,
}

MANUAL_FILTERS = {key: value for key, value in MANUAL_FILTERS.items() if value is not None}

print("Manual search configured.")
print("Query:", MANUAL_SEARCH_QUERY)
print("Top K:", MANUAL_TOP_K)
print("Filters:", MANUAL_FILTERS)

Manual search configured.
Query: How much revenue did the company earn and what were the major expenses?
Top K: 5
Filters: {}


## 12. Run the Manual Search

In [18]:
manual_search_run = execute_vector_search(
    query_text=MANUAL_SEARCH_QUERY,
    top_k=MANUAL_TOP_K,
    filters=MANUAL_FILTERS,
    scenario_name="manual_search",
    scenario_description="Single interactive search for quick debugging.",
)

print("Manual search completed successfully.")
print("Result count:", manual_search_run["summary"]["result_count"])
print("Top score:", manual_search_run["summary"]["top_score"])
print("Average score:", manual_search_run["summary"]["average_score"])

Manual search completed successfully.
Result count: 5
Top score: 0.338993
Average score: 0.321902


## 13. Inspect Manual Search Results

In [19]:
print_search_results(manual_search_run, max_results=MANUAL_TOP_K)

Scenario: manual_search
Description: Single interactive search for quick debugging.
Query: How much revenue did the company earn and what were the major expenses?
Top K: 5
Filters: {}
Embedding latency (s): 0.3794
Search latency (s): 0.0735
Result count: 5
Top score: 0.338993
----------------------------------------------------------------------------------------------------
Rank: 1 | Score: 0.338993
Company Name: Max Healthcare Institute Limited
Taxonomy: COMMERCIAL_NBFC
Section: Income Statement & Profitability
Chunk Type: raw_fact_group
Reporting Period: 2024-10-01 to 2024-12-31
Context Type: Duration
Chunk ID: indas-118123-1365717-30012025030002-32d59527d66b
Source File: INDAS_118123_1365717_30012025030002.xml
Unified Fields: []
Source Tags: ['DescriptionOfOtherExpenses', 'OtherExpenses']
Text Preview:
Company Name: Max Healthcare Institute Limited
Taxonomy: COMMERCIAL_NBFC
Report Type: Consolidated
Reporting Period: 2024-10-01 to 2024-12-31
Context Type: Duration
Section: Income S

## 14. Define Retrieval Evaluation Scenarios

In [20]:
retrieval_scenarios: List[Dict[str, Any]] = [
    {
        "scenario_name": "baseline_profitability_all_chunks",
        "description": "Baseline search across all chunk types without metadata filters.",
        "query_text": "How much revenue did the company earn and what were the major expenses?",
        "top_k": 5,
        "filters": {},
    },
    {
        "scenario_name": "profitability_unified_only",
        "description": "Profitability search restricted to unified metric chunks.",
        "query_text": "How much revenue did the company earn and what were the major expenses?",
        "top_k": 5,
        "filters": {
            "chunk_type": "unified_metric_group",
            "is_unified_chunk": True,
        },
    },
    {
        "scenario_name": "income_statement_section_only",
        "description": "Profitability search restricted to income statement related chunks.",
        "query_text": "What were the company's total income, direct expenses, and net profit?",
        "top_k": 5,
        "filters": {
            "section_name": "Income Statement & Profitability",
        },
    },
    {
        "scenario_name": "capital_eps_query",
        "description": "Search for EPS and capital-related information.",
        "query_text": "What is the basic EPS and what capital related information is reported?",
        "top_k": 5,
        "filters": {
            "section_name": "Capital, EPS & Ratios",
        },
    },
    {
        "scenario_name": "cashflow_operating_query",
        "description": "Search for cash flow from operating activities.",
        "query_text": "What were the cash flows from operating activities?",
        "top_k": 5,
        "filters": {
            "section_name": "Cash Flow Statement",
        },
    },
    {
        "scenario_name": "balance_sheet_instant_query",
        "description": "Search for instant-period balance sheet style results.",
        "query_text": "What were the key assets and liabilities as of the reporting date?",
        "top_k": 5,
        "filters": {
            "context_type": "Instant",
        },
    },
    {
        "scenario_name": "segment_dimension_query",
        "description": "Search for segment disclosures and dimension-rich records.",
        "query_text": "How did the company's business segments perform and were there segment assets or liabilities?",
        "top_k": 5,
        "filters": {
            "section_name": "Segment Performance",
            "has_dimensions": True,
        },
    },
    {
        "scenario_name": "narrative_disclosure_query",
        "description": "Search for narrative disclosure and note-style chunks.",
        "query_text": "What accounting policies, judgments, or important disclosures are described in the filing?",
        "top_k": 5,
        "filters": {
            "chunk_type": "narrative_disclosure",
            "is_narrative_chunk": True,
        },
    },
    {
        "scenario_name": "banking_interest_query",
        "description": "Search for bank-style income language like interest earned and interest expended.",
        "query_text": "What interest income was earned and what interest expenses were reported?",
        "top_k": 5,
        "filters": {},
    },
    {
        "scenario_name": "insurance_premium_claims_query",
        "description": "Search for insurance-style premium and claims language.",
        "query_text": "What premiums were earned and what claims were incurred or paid?",
        "top_k": 5,
        "filters": {},
    },
]

filters_to_inject = {
    "company_name": CONFIG.get("demo_company_name_filter"),
    "taxonomy": CONFIG.get("demo_taxonomy_filter")
}

active_filters = {key: value for key, value in filters_to_inject.items() if value}

if active_filters:
    print(f"\nInjecting global filters: {active_filters} into all scenarios...\n")
    
    for scenario in retrieval_scenarios:
        scenario["filters"].update(active_filters)


print("Retrieval scenarios defined successfully.")
print("Total scenarios:", len(retrieval_scenarios))

Retrieval scenarios defined successfully.
Total scenarios: 10


## 15. Review Retrieval Scenario Plan

In [21]:
for scenario in retrieval_scenarios:
    print("-" * 100)
    print("Scenario Name :", scenario["scenario_name"])
    print("Description   :", scenario["description"])
    print("Query         :", scenario["query_text"])
    print("Top K         :", scenario["top_k"])
    print("Filters       :", json.dumps(scenario["filters"], ensure_ascii=False))

----------------------------------------------------------------------------------------------------
Scenario Name : baseline_profitability_all_chunks
Description   : Baseline search across all chunk types without metadata filters.
Query         : How much revenue did the company earn and what were the major expenses?
Top K         : 5
Filters       : {}
----------------------------------------------------------------------------------------------------
Scenario Name : profitability_unified_only
Description   : Profitability search restricted to unified metric chunks.
Query         : How much revenue did the company earn and what were the major expenses?
Top K         : 5
Filters       : {"chunk_type": "unified_metric_group", "is_unified_chunk": true}
----------------------------------------------------------------------------------------------------
Scenario Name : income_statement_section_only
Description   : Profitability search restricted to income statement related chunks.
Query  

## 16. Run Retrieval Evaluation Across All Scenarios

In [22]:
scenario_runs: List[Dict[str, Any]] = []
scenario_failures: List[Dict[str, Any]] = []

print(f"Running retrieval evaluation for {len(retrieval_scenarios)} scenarios...\n")

for scenario in retrieval_scenarios:
    try:
        run_output = execute_vector_search(
            query_text=scenario["query_text"],
            top_k=scenario.get("top_k", CONFIG["default_top_k"]),
            filters=scenario.get("filters", {}),
            scenario_name=scenario["scenario_name"],
            scenario_description=scenario.get("description"),
        )
        scenario_runs.append(run_output)

        print(f"Completed: {scenario['scenario_name']}")
        print(f"  Results   : {run_output['summary']['result_count']}")
        print(f"  Top score : {run_output['summary']['top_score']}")
        print(f"  Avg score : {run_output['summary']['average_score']}")
        print(f"  Search sec: {run_output['search_meta']['search_latency_seconds']}")
        print("-" * 80)

    except Exception as e:
        scenario_failures.append({
            "scenario_name": scenario["scenario_name"],
            "query_text": scenario["query_text"],
            "filters": scenario.get("filters", {}),
            "error": str(e),
        })
        print(f"Failed: {scenario['scenario_name']} -> {e}")
        print("-" * 80)

print("Scenario evaluation complete.")
print("Successful scenarios:", len(scenario_runs))
print("Failed scenarios    :", len(scenario_failures))

Running retrieval evaluation for 10 scenarios...

Completed: baseline_profitability_all_chunks
  Results   : 5
  Top score : 0.338993
  Avg score : 0.321902
  Search sec: 0.0386
--------------------------------------------------------------------------------
Completed: profitability_unified_only
  Results   : 5
  Top score : 0.263308
  Avg score : 0.253874
  Search sec: 0.0777
--------------------------------------------------------------------------------
Completed: income_statement_section_only
  Results   : 5
  Top score : 0.280402
  Avg score : 0.271605
  Search sec: 0.0692
--------------------------------------------------------------------------------
Completed: capital_eps_query
  Results   : 5
  Top score : 0.343725
  Avg score : 0.316295
  Search sec: 0.0547
--------------------------------------------------------------------------------
Completed: cashflow_operating_query
  Results   : 5
  Top score : 0.384156
  Avg score : 0.326479
  Search sec: 0.0524
----------------------

## 17. Inspect Sample Scenario Results

In [23]:
preview_runs = min(3, len(scenario_runs))

for i in range(preview_runs):
    print_search_results(scenario_runs[i], max_results=3)
    print("\n")

Scenario: baseline_profitability_all_chunks
Description: Baseline search across all chunk types without metadata filters.
Query: How much revenue did the company earn and what were the major expenses?
Top K: 5
Filters: {}
Embedding latency (s): 0.3802
Search latency (s): 0.0386
Result count: 5
Top score: 0.338993
----------------------------------------------------------------------------------------------------
Rank: 1 | Score: 0.338993
Company Name: Max Healthcare Institute Limited
Taxonomy: COMMERCIAL_NBFC
Section: Income Statement & Profitability
Chunk Type: raw_fact_group
Reporting Period: 2024-10-01 to 2024-12-31
Context Type: Duration
Chunk ID: indas-118123-1365717-30012025030002-32d59527d66b
Source File: INDAS_118123_1365717_30012025030002.xml
Unified Fields: []
Source Tags: ['DescriptionOfOtherExpenses', 'OtherExpenses']
Text Preview:
Company Name: Max Healthcare Institute Limited
Taxonomy: COMMERCIAL_NBFC
Report Type: Consolidated
Reporting Period: 2024-10-01 to 2024-12-31
Co

## 18. Build Scenario-Level Retrieval Summary Table

In [24]:
scenario_summary_records: List[Dict[str, Any]] = []

for run in scenario_runs:
    record = {
        "scenario_name": run["scenario_name"],
        "scenario_description": run.get("scenario_description"),
        "query_text": run["query_text"],
        "top_k": run["top_k"],
        "filters": run["filters"],
        "result_count": run["summary"]["result_count"],
        "top_score": run["summary"]["top_score"],
        "lowest_score": run["summary"]["lowest_score"],
        "average_score": run["summary"]["average_score"],
        "full_filter_match_count": run["summary"]["full_filter_match_count"],
        "top_result_company_name": run["summary"]["top_result_company_name"],
        "top_result_section_name": run["summary"]["top_result_section_name"],
        "top_result_chunk_type": run["summary"]["top_result_chunk_type"],
        "top_result_taxonomy": run["summary"]["top_result_taxonomy"],
        "embedding_latency_seconds": run["embedding_meta"]["latency_seconds"],
        "search_latency_seconds": run["search_meta"]["search_latency_seconds"],
        "embedding_attempts_used": run["embedding_meta"]["attempts_used"],
        "embedding_throttled_retries": run["embedding_meta"]["throttled_retries"],
    }
    scenario_summary_records.append(record)

print("Scenario summary records prepared.")
print("Total summary rows:", len(scenario_summary_records))

for row in scenario_summary_records[:5]:
    print(json.dumps(row, indent=2, ensure_ascii=False))

Scenario summary records prepared.
Total summary rows: 10
{
  "scenario_name": "baseline_profitability_all_chunks",
  "scenario_description": "Baseline search across all chunk types without metadata filters.",
  "query_text": "How much revenue did the company earn and what were the major expenses?",
  "top_k": 5,
  "filters": {},
  "result_count": 5,
  "top_score": 0.338993,
  "lowest_score": 0.310253,
  "average_score": 0.321902,
  "full_filter_match_count": 5,
  "top_result_company_name": "Max Healthcare Institute Limited",
  "top_result_section_name": "Income Statement & Profitability",
  "top_result_chunk_type": "raw_fact_group",
  "top_result_taxonomy": "COMMERCIAL_NBFC",
  "embedding_latency_seconds": 0.3802,
  "search_latency_seconds": 0.0386,
  "embedding_attempts_used": 1,
  "embedding_throttled_retries": 0
}
{
  "scenario_name": "profitability_unified_only",
  "scenario_description": "Profitability search restricted to unified metric chunks.",
  "query_text": "How much revenu

## 19. Analyze Retrieved Result Distribution by Chunk Type, Section, and Taxonomy

In [25]:
retrieved_chunk_type_counter = Counter()
retrieved_section_counter = Counter()
retrieved_taxonomy_counter = Counter()
retrieved_company_counter = Counter()

for run in scenario_runs:
    for result in run["results"]:
        retrieved_chunk_type_counter[result.get("chunk_type", "Unknown")] += 1
        retrieved_section_counter[result.get("section_name", "Unknown")] += 1
        retrieved_taxonomy_counter[result.get("taxonomy", "UNKNOWN")] += 1
        retrieved_company_counter[result.get("company_name", "Unknown Company")] += 1

print("Retrieved result distribution")
print("-" * 100)

print("\nBy chunk type:")
for key, value in retrieved_chunk_type_counter.most_common():
    print(f"- {key}: {value}")

print("\nBy section name:")
for key, value in retrieved_section_counter.most_common():
    print(f"- {key}: {value}")

print("\nBy taxonomy:")
for key, value in retrieved_taxonomy_counter.most_common():
    print(f"- {key}: {value}")

print("\nBy company name:")
for key, value in retrieved_company_counter.most_common(20):
    print(f"- {key}: {value}")

Retrieved result distribution
----------------------------------------------------------------------------------------------------

By chunk type:
- raw_fact_group: 28
- narrative_disclosure: 12
- unified_metric_group: 10

By section name:
- Income Statement & Profitability: 13
- Narrative Notes & Disclosures: 12
- Unified Financial Summary: 5
- Capital, EPS & Ratios: 5
- Cash Flow Statement: 5
- Balance Sheet – Assets: 5
- Segment Performance: 5

By taxonomy:
- INSURANCE: 19
- COMMERCIAL_NBFC: 17
- BANK: 14

By company name:
- RELIANCE INDUSTRIES LIMITED: 15
- Max Healthcare Institute Limited: 12
- ICICI Lombard General Insurance Company Limited: 12
- HDFC Life Insurance Company Limited: 7
- Bajaj Finserv Limited: 4


## 20. Check Metadata Filter Alignment Across Scenario Results

In [26]:
alignment_summary_records: List[Dict[str, Any]] = []

for run in scenario_runs:
    total_results = len(run["results"])
    matched_results = sum(
        1 for item in run["results"]
        if item.get("filter_alignment", {}).get("all_filters_matched") is True
    )

    alignment_summary_records.append({
        "scenario_name": run["scenario_name"],
        "filters": run["filters"],
        "total_results": total_results,
        "fully_matched_results": matched_results,
        "full_match_rate": safe_round((matched_results / total_results), 4) if total_results else None,
    })

print("Metadata alignment summary")
print("-" * 100)
for item in alignment_summary_records:
    print(json.dumps(item, indent=2, ensure_ascii=False))

Metadata alignment summary
----------------------------------------------------------------------------------------------------
{
  "scenario_name": "baseline_profitability_all_chunks",
  "filters": {},
  "total_results": 5,
  "fully_matched_results": 5,
  "full_match_rate": 1.0
}
{
  "scenario_name": "profitability_unified_only",
  "filters": {
    "chunk_type": "unified_metric_group",
    "is_unified_chunk": true
  },
  "total_results": 5,
  "fully_matched_results": 5,
  "full_match_rate": 1.0
}
{
  "scenario_name": "income_statement_section_only",
  "filters": {
    "section_name": "Income Statement & Profitability"
  },
  "total_results": 5,
  "fully_matched_results": 5,
  "full_match_rate": 1.0
}
{
  "scenario_name": "capital_eps_query",
  "filters": {
    "section_name": "Capital, EPS & Ratios"
  },
  "total_results": 5,
  "fully_matched_results": 5,
  "full_match_rate": 1.0
}
{
  "scenario_name": "cashflow_operating_query",
  "filters": {
    "section_name": "Cash Flow Statement

## 21. Flag Low-Confidence and Zero-Result Scenarios

In [27]:
zero_result_scenarios = []
low_confidence_scenarios = []

for run in scenario_runs:
    result_count = run["summary"]["result_count"]
    top_score = run["summary"]["top_score"]

    if result_count == 0:
        zero_result_scenarios.append({
            "scenario_name": run["scenario_name"],
            "query_text": run["query_text"],
            "filters": run["filters"],
        })

    if top_score is not None and top_score < CONFIG["low_score_warning_threshold"]:
        low_confidence_scenarios.append({
            "scenario_name": run["scenario_name"],
            "query_text": run["query_text"],
            "top_score": top_score,
            "filters": run["filters"],
        })

print("Zero-result scenarios:", len(zero_result_scenarios))
if zero_result_scenarios:
    for item in zero_result_scenarios:
        print(json.dumps(item, indent=2, ensure_ascii=False))

print("\nLow-confidence scenarios:", len(low_confidence_scenarios))
if low_confidence_scenarios:
    for item in low_confidence_scenarios:
        print(json.dumps(item, indent=2, ensure_ascii=False))

Zero-result scenarios: 0

Low-confidence scenarios: 10
{
  "scenario_name": "baseline_profitability_all_chunks",
  "query_text": "How much revenue did the company earn and what were the major expenses?",
  "top_score": 0.338993,
  "filters": {}
}
{
  "scenario_name": "profitability_unified_only",
  "query_text": "How much revenue did the company earn and what were the major expenses?",
  "top_score": 0.263308,
  "filters": {
    "chunk_type": "unified_metric_group",
    "is_unified_chunk": true
  }
}
{
  "scenario_name": "income_statement_section_only",
  "query_text": "What were the company's total income, direct expenses, and net profit?",
  "top_score": 0.280402,
  "filters": {
    "section_name": "Income Statement & Profitability"
  }
}
{
  "scenario_name": "capital_eps_query",
  "query_text": "What is the basic EPS and what capital related information is reported?",
  "top_score": 0.343725,
  "filters": {
    "section_name": "Capital, EPS & Ratios"
  }
}
{
  "scenario_name": "cash

## 22. Build Final Retrieval Evaluation Payloads

In [28]:
average_search_latency = safe_round(
    sum(run["search_meta"]["search_latency_seconds"] for run in scenario_runs) / len(scenario_runs),
    4
) if scenario_runs else None

average_embedding_latency = safe_round(
    sum(run["embedding_meta"]["latency_seconds"] for run in scenario_runs) / len(scenario_runs),
    4
) if scenario_runs else None

final_retrieval_summary = {
    "pipeline_version": "retrieval_evaluation_v2",
    "collection_summary": collection_summary,
    "config": CONFIG,
    "manual_search": {
        "query_text": manual_search_run["query_text"],
        "filters": manual_search_run["filters"],
        "result_count": manual_search_run["summary"]["result_count"],
        "top_score": manual_search_run["summary"]["top_score"],
        "average_score": manual_search_run["summary"]["average_score"],
    },
    "scenario_evaluation": {
        "total_scenarios_defined": len(retrieval_scenarios),
        "successful_scenarios": len(scenario_runs),
        "failed_scenarios": len(scenario_failures),
        "zero_result_scenarios": len(zero_result_scenarios),
        "low_confidence_scenarios": len(low_confidence_scenarios),
        "average_search_latency_seconds": average_search_latency,
        "average_embedding_latency_seconds": average_embedding_latency,
    },
    "retrieved_distribution": {
        "chunk_type": dict(retrieved_chunk_type_counter),
        "section_name": dict(retrieved_section_counter),
        "taxonomy": dict(retrieved_taxonomy_counter),
        "company_name": dict(retrieved_company_counter),
    },
    "alignment_summary": alignment_summary_records,
    "scenario_failures": scenario_failures,
}

print("Final retrieval summary prepared.")
print(json.dumps(final_retrieval_summary["scenario_evaluation"], indent=2, ensure_ascii=False))

Final retrieval summary prepared.
{
  "total_scenarios_defined": 10,
  "successful_scenarios": 10,
  "failed_scenarios": 0,
  "zero_result_scenarios": 0,
  "low_confidence_scenarios": 10,
  "average_search_latency_seconds": 0.0507,
  "average_embedding_latency_seconds": 0.5407
}


## 23. Save Retrieval Evaluation Outputs

In [29]:
manual_search_output_path = save_json(
    manual_search_run,
    Path(CONFIG["manual_search_output_path"]),
)

scenario_results_output_path = save_json(
    {
        "scenario_runs": scenario_runs,
        "scenario_failures": scenario_failures,
        "zero_result_scenarios": zero_result_scenarios,
        "low_confidence_scenarios": low_confidence_scenarios,
    },
    Path(CONFIG["scenario_results_output_path"]),
)

scenario_summary_output_path = save_json(
    {
        "summary_records": scenario_summary_records,
        "final_retrieval_summary": final_retrieval_summary,
    },
    Path(CONFIG["scenario_summary_output_path"]),
)

print("Retrieval evaluation outputs saved successfully.")
print("Manual search output   :", manual_search_output_path.resolve())
print("Scenario results output:", scenario_results_output_path.resolve())
print("Scenario summary output:", scenario_summary_output_path.resolve())

Retrieval evaluation outputs saved successfully.
Manual search output   : C:\Users\Home\llmops-nse-rag\data\processed\audits\retrieval_manual_search_v1.json
Scenario results output: C:\Users\Home\llmops-nse-rag\data\processed\audits\retrieval_scenario_results_v1.json
Scenario summary output: C:\Users\Home\llmops-nse-rag\data\processed\audits\retrieval_scenario_summary_v1.json


## 24. Print Final Retrieval Evaluation Summary

In [30]:
print("Final retrieval evaluation summary")
print("-" * 100)
print("Collection name                  :", CONFIG["collection_name"])
print("Manual search result count       :", manual_search_run["summary"]["result_count"])
print("Total scenarios defined          :", len(retrieval_scenarios))
print("Successful scenarios             :", len(scenario_runs))
print("Failed scenarios                 :", len(scenario_failures))
print("Zero-result scenarios            :", len(zero_result_scenarios))
print("Low-confidence scenarios         :", len(low_confidence_scenarios))
print("Average embedding latency (sec)  :", average_embedding_latency)
print("Average search latency (sec)     :", average_search_latency)
print("Manual search output file        :", manual_search_output_path.name)
print("Scenario results output file     :", scenario_results_output_path.name)
print("Scenario summary output file     :", scenario_summary_output_path.name)

Final retrieval evaluation summary
----------------------------------------------------------------------------------------------------
Collection name                  : financial_chunks_xbrl_context_aware_v1
Manual search result count       : 5
Total scenarios defined          : 10
Successful scenarios             : 10
Failed scenarios                 : 0
Zero-result scenarios            : 0
Low-confidence scenarios         : 10
Average embedding latency (sec)  : 0.5407
Average search latency (sec)     : 0.0507
Manual search output file        : retrieval_manual_search_v1.json
Scenario results output file     : retrieval_scenario_results_v1.json
Scenario summary output file     : retrieval_scenario_summary_v1.json
